# CNN + BiLSTM + Attention — SEED-IV (Trial-Grouped Split, Per-Subject Normalization)
Loads directly from your cleaned CSV. Groups consecutive samples within each trial into sliding windows so the BiLSTM sees a real temporal sequence.

Changes vs. the previous version:
- **Per-subject normalization**: EEG baselines vary a lot between people. Z-scoring each subject's features using *that subject's own* mean/std removes most of this inter-subject offset, so the model can't shortcut by learning subject identity instead of emotion-relevant patterns. (This replaces the single global `StandardScaler`.)
- **Smaller model** (`cnn_ch=128`, `lstm_h=64`) — the previous 256/128 config memorized the training trials very quickly (98% train acc by epoch 25) while validation stayed around 45-50%.
- **Stronger regularization**: `DROPOUT=0.6`, `WEIGHT_DECAY=1e-3`, lower `LR=5e-4`.

Trial groups still kept out of both train and test via `GroupShuffleSplit`.

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report



Device: cpu


In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Hyperparameters ───────────────────────────────────────────
WINDOW_SIZE  = 10      # consecutive samples per sequence fed to BiLSTM
STEP_SIZE    = 5       # sliding window stride
BATCH_SIZE   = 64
EPOCHS       = 80
LR           = 5e-4    # lowered from 1e-3 — training acc was climbing too fast
DROPOUT      = 0.6     # raised from 0.5
WEIGHT_DECAY = 1e-3     # raised from 1e-4
PATIENCE     = 15      # early stopping

In [ ]:
# ── Load CSV ─────────────────────────────────────────────────
df = pd.read_csv("../data/neurosense_cleaned.csv")
print(f'Loaded: {df.shape}  |  Label distribution:\n{df["label"].value_counts().sort_index()}')

# ── Feature columns ────────────────────────────────────────────
EEG_COLS  = [f'eeg_feature_{i}' for i in range(1, 311)]   # 310 features
EYE_COLS  = [f'eye_feature_{i}' for i in range(1, 32)]    # 31 features
STAT_COLS = ['avg_eeg','std_eeg','max_eeg','min_eeg',
             'avg_eye','std_eye','max_eye','min_eye',
             'eeg_eye_ratio','eeg_range','eye_range',
             'eeg_stability','eye_stability','phys_activity']
FEAT_COLS = EEG_COLS + EYE_COLS + STAT_COLS   # 355 total
META_COLS = ['subject', 'session', 'trial']

INPUT_SIZE  = len(FEAT_COLS)   # 355
NUM_CLASSES = df['label'].nunique()
print(f'Features: {INPUT_SIZE}  |  Classes: {NUM_CLASSES}  |  Subjects: {df["subject"].nunique()}')

# ── Per-subject z-score normalization ──────────────────────────
# Each subject's features are scaled using that subject's own mean/std.
# This removes inter-subject baseline differences while preserving the
# relative (emotion-driven) variation within each subject. We deliberately
# do this BEFORE the train/test split: per-subject stats are a property of
# the subject, not the label, so this is a calibration step rather than a
# label-leakage concern.
df[FEAT_COLS] = (
    df.groupby('subject')[FEAT_COLS]
      .transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
)

print('Per-subject normalization applied.')
print(df[FEAT_COLS].describe().loc[['mean', 'std']].iloc[:, :4])

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/seed-iv-dataset-csv-format/neurosense_cleaned.csv'

## 3. Build Windows (Tagged with Trial Group IDs)

In [ ]:
def build_windows(df, feat_cols, window_size, step_size):
    """
    Group consecutive rows within each (subject, session, trial)
    into sliding windows so the BiLSTM sees a temporal sequence.

    Every window is tagged with a group id identifying which trial
    it came from. This lets us later split train/test by trial so
    overlapping windows from the same trial never end up on both
    sides of the split.

    Returns:
        X      : (N, window_size, n_features)  float32
        y      : (N,)                          int64
        groups : (N,)                          int64  (trial group id)
    """
    X_list, y_list, g_list = [], [], []
    grouped = df.groupby(['subject', 'session', 'trial'], sort=False)

    for group_id, (_, grp) in enumerate(grouped):
        feats = grp[feat_cols].values.astype(np.float32)  # (T, F)
        label = int(grp['label'].iloc[0])
        T = len(feats)

        if T < window_size:
            # Pad short trials by repeating the last row
            pad = np.repeat(feats[-1:], window_size - T, axis=0)
            feats = np.vstack([feats, pad])
            T = window_size

        for start in range(0, T - window_size + 1, step_size):
            X_list.append(feats[start : start + window_size])
            y_list.append(label)
            g_list.append(group_id)

    return (np.array(X_list, dtype=np.float32),
            np.array(y_list, dtype=np.int64),
            np.array(g_list, dtype=np.int64))


X, y, groups = build_windows(df, FEAT_COLS, WINDOW_SIZE, STEP_SIZE)
print(f'Windows — X: {X.shape}  y: {y.shape}  groups: {groups.shape}')
print(f'Number of trials (groups): {len(np.unique(groups))}')

unique, counts = np.unique(y, return_counts=True)
emotions = {0:'Neutral', 1:'Sad', 2:'Fear', 3:'Happy'}
for u, c in zip(unique, counts):
    print(f'  {emotions[u]:>8}: {c}')

## 4. Split (Trial-Grouped)
No additional global scaler is applied here — features were already z-scored per subject in step 2. Applying a global `StandardScaler` on top would re-introduce the inter-subject scale differences we just removed.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Sanity check: no trial's windows should appear on both sides
train_groups = set(groups[train_idx])
test_groups  = set(groups[test_idx])
assert len(train_groups & test_groups) == 0, 'Trial leakage detected!'

def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=True)

train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
test_loader  = make_loader(X_test,  y_test,  BATCH_SIZE, shuffle=False)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train trials: {len(train_groups)}  |  Test trials: {len(test_groups)}')

## 5. Model: CNN + BiLSTM + Attention
Reduced capacity (`cnn_ch=128`, `lstm_h=64`, down from 256/128) — the larger model was memorizing the training trials almost perfectly (98% train acc) well before validation accuracy stabilized.

In [ ]:
class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.w = nn.Linear(dim, 1, bias=False)

    def forward(self, x):                                       # (B, T, H)
        w = torch.softmax(self.w(x).squeeze(-1), dim=1)        # (B, T)
        return (w.unsqueeze(1) @ x).squeeze(1)                 # (B, H)


class CNN_BiLSTM(nn.Module):
    def __init__(self, input_size, cnn_ch=128, lstm_h=64,
                 lstm_layers=2, num_classes=4, dropout=0.6):
        super().__init__()

        # CNN: extracts local patterns across time, projects features down
        self.cnn = nn.Sequential(
            nn.Conv1d(input_size, cnn_ch, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_ch), nn.GELU(), nn.Dropout(dropout * 0.4),
            nn.Conv1d(cnn_ch, cnn_ch, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_ch), nn.GELU(), nn.Dropout(dropout * 0.4),
        )

        # BiLSTM: captures long-range temporal dependencies
        self.bilstm = nn.LSTM(
            cnn_ch, lstm_h, num_layers=lstm_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        # Attention: soft-weights which timesteps matter most
        self.attn = Attention(lstm_h * 2)

        # Classifier head
        self.head = nn.Sequential(
            nn.Linear(lstm_h * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):                          # x: (B, T, F)
        x = self.cnn(x.permute(0, 2, 1))          # (B, cnn_ch, T)
        x, _ = self.bilstm(x.permute(0, 2, 1))    # (B, T, lstm_h*2)
        x = self.attn(x)                           # (B, lstm_h*2)
        return self.head(x)                        # (B, num_classes)


model = CNN_BiLSTM(
    input_size=INPUT_SIZE,
    cnn_ch=128,
    lstm_h=64,
    lstm_layers=2,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

## 6. Training

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_acc, best_state, no_improve = 0.0, None, 0

print(f'{"Ep":>4}  {"Tr Loss":>8}  {"Tr Acc":>7}  {"Val Acc":>7}')
print('-' * 35)

for ep in range(1, EPOCHS + 1):

    # ── Train ─────────────────────────────────────────────────
    model.train()
    tr_loss = tr_correct = tr_total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_loss    += loss.item() * xb.size(0)
        tr_correct += (logits.argmax(1) == yb).sum().item()
        tr_total   += xb.size(0)
    scheduler.step()

    # ── Validate ─────────────────────────────────────────────
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
            labels.extend(yb.numpy())
    val_acc = accuracy_score(labels, preds)

    if val_acc > best_acc:
        best_acc   = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    if ep % 5 == 0 or ep == 1:
        print(f'{ep:>4}  {tr_loss/tr_total:>8.4f}  '
              f'{tr_correct/tr_total*100:>6.2f}%  {val_acc*100:>6.2f}%')

    if no_improve >= PATIENCE:
        print(f'Early stop at epoch {ep}  |  Best val acc: {best_acc*100:.2f}%')
        break

print(f'\nBest Validation Accuracy: {best_acc*100:.2f}%')

## 7. Final Evaluation

In [ ]:
model.load_state_dict(best_state)
model.eval().to(DEVICE)

preds, labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
        labels.extend(yb.numpy())

print(f'Test Accuracy : {accuracy_score(labels, preds)*100:.2f}%')
print(f'Weighted F1   : {f1_score(labels, preds, average="weighted"):.4f}')
print()
print(classification_report(
    labels, preds,
    target_names=[emotions[i] for i in range(NUM_CLASSES)]
))

## 8. Save

In [ ]:
torch.save(best_state, 'cnn_bilstm_seed_iv_groupsplit_v2.pth')
print('Saved: cnn_bilstm_seed_iv_groupsplit_v2.pth')